# 04. 知名度指標の立て直し：自作ラベルで推薦対象を絞る

## 経緯

`03_awareness.ipynb` でWikipedia日本語版の記事有無を子どもの認知度の代理指標として
検証したところ、50職業のラベリングとの一致率は50.0%（偶然と同水準）で、ズレには
明確な方向性があった。Wikipediaの記事有無は「行政的・制度的に確立した専門職かどうか」
を測っており、「子どもの日常での接触頻度」とは別の軸だった。記事有無を主指標にする
計画は棄却した。

検索ボリューム・教科書出現・求人掲載数もそれぞれ別のものを測っており、同じ壁に
当たることが分かっている（`docs/design.md`参照）。**「子どもの認知度」を測った公開
データはそもそも存在しない。認知度は職業単体の属性ではなく、子どもと職業の関係だから
である。** 代わりを探し続けるのではなく、自分でラベルを作ってモデルで拡張する方針に
切り替える。

## ラベルの位置づけ（重要な限界）

以下で付けるラベルは、job tag・教育データに関する知識を持つ**1名の評価者による、
一般的な日本の子ども向けメディア・生活文化についての常識的判断に基づく代理ラベル**
である。実際の生徒への調査や、教育現場での実地の経験に基づく判定ではない。
変数名・UI表示ともに「推定認知度（ラベル作成者1名による）」であることを明示し、
実測の認知度であるかのように扱わない。


In [1]:
import re

import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler

from ipd_loader import load_description, load_numeric

pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 200)

desc, desc_labels = load_description()
num, num_labels = load_numeric()
names = desc[desc.columns[1]]
print("職業数:", len(names))


職業数: 556


## ラベル定義（固定）

**「中学生が、この職業名を見聞きしたとき、仕事内容を1文で説明できそうか」** を基準に、
次の3段階で判定する。

- **知っている（2点）**: 名前を見て、仕事内容を自分の言葉で1文程度説明できそうな職業。
  日常的な接触機会がある、メディアでの露出が多い、学校教育で扱われる、など
- **名前は聞いたことがある（1点）**: 名前や存在は聞いたことがありそうだが、具体的に
  何をする仕事か1文では説明できなさそうな職業
- **知らない（0点）**: 名前を見ても、何をする仕事か見当がつかなさそうな職業。専門資格・
  業界特化の技術職・工程区分など

## 層化抽出

556職業を、主な厚労省編職業分類の大分類コード（先頭2桁、15区分）で層化し、
各区分の母数に比例して合計179件を抽出する。


In [2]:
def major_code(v):
    if not isinstance(v, str):
        return None
    m = re.match(r"^(\d+)", v)
    return m.group(1) if m else None


major = desc["IPD_02_02_000"].apply(major_code)
TARGET_N = 180
counts = major.value_counts().sort_index()
alloc = (counts * TARGET_N / counts.sum()).round().astype(int).clip(lower=1)

rng_state = 7
sample_idx = []
for code, n_take in alloc.items():
    pool = pd.Index(desc.index[major == code])
    n_take = min(n_take, len(pool))
    picked = pool.to_series().sample(n=n_take, random_state=rng_state).index
    sample_idx.extend(picked)

sample_idx = sorted(sample_idx)
print("抽出件数:", len(sample_idx), "（大分類15区分に比例配分）")


抽出件数: 179 （大分類15区分に比例配分）


## ラベル付け（179件）

In [3]:
label_order = ["知らない", "名前は聞いたことがある", "知っている"]

# 収録番号(行位置) -> 認知度スコア(0/1/2)。判定基準は上のセルの定義に固定。
awareness_scores = {
    2: 2, 11: 0, 12: 1, 13: 1, 15: 0, 17: 0, 18: 0, 21: 0,
    25: 2, 26: 0, 27: 0, 28: 0, 32: 1, 33: 0, 41: 0, 48: 0,
    49: 1, 51: 2, 53: 2, 54: 2, 61: 0, 66: 2, 70: 2, 71: 2,
    72: 0, 75: 2, 80: 1, 89: 0, 90: 1, 91: 0, 93: 0, 94: 1,
    97: 2, 105: 2, 111: 1, 115: 2, 116: 2, 118: 2, 119: 2, 120: 2,
    121: 0, 125: 1, 126: 2, 130: 1, 134: 0, 135: 0, 140: 1, 142: 2,
    144: 0, 150: 2, 153: 2, 155: 2, 156: 2, 159: 0, 163: 0, 167: 0,
    169: 2, 173: 1, 175: 2, 182: 0, 184: 2, 188: 0, 190: 2, 191: 2,
    193: 0, 195: 2, 197: 0, 198: 2, 207: 1, 208: 2, 210: 1, 213: 2,
    215: 0, 216: 1, 219: 1, 220: 2, 223: 2, 229: 0, 231: 0, 236: 1,
    237: 0, 238: 2, 242: 0, 245: 0, 246: 0, 248: 0, 251: 0, 259: 0,
    260: 0, 261: 1, 263: 0, 266: 0, 267: 1, 268: 0, 270: 1, 271: 0,
    276: 0, 277: 1, 278: 0, 286: 1, 290: 1, 292: 1, 293: 1, 295: 0,
    299: 0, 300: 0, 303: 0, 304: 0, 305: 2, 306: 1, 307: 1, 315: 0,
    317: 0, 319: 0, 323: 2, 327: 1, 328: 0, 332: 2, 340: 0, 341: 2,
    342: 0, 344: 2, 351: 2, 355: 0, 358: 0, 359: 0, 363: 0, 364: 1,
    373: 0, 376: 2, 378: 2, 383: 1, 390: 2, 391: 2, 395: 0, 400: 2,
    402: 1, 409: 0, 410: 0, 413: 0, 414: 0, 416: 1, 420: 0, 426: 0,
    427: 1, 432: 0, 433: 0, 438: 2, 440: 0, 441: 1, 443: 0, 446: 2,
    448: 2, 449: 1, 454: 2, 455: 2, 457: 0, 460: 1, 462: 0, 463: 2,
    464: 2, 467: 0, 477: 2, 478: 1, 483: 2, 487: 0, 499: 1, 502: 0,
    504: 0, 508: 1, 511: 0, 526: 0, 530: 0, 531: 0, 535: 0, 544: 2,
    546: 2, 553: 2, 555: 1,
}

assert len(awareness_scores) == len(sample_idx)
assert set(awareness_scores) == set(sample_idx)

label_df = pd.DataFrame({
    "職業名": names.loc[sample_idx],
    "awareness_score": pd.Series(awareness_scores),
})
label_df["awareness_label"] = label_df["awareness_score"].map(dict(enumerate(label_order)))
label_df["awareness_label"].value_counts().reindex(label_order)

awareness_label
知らない           82
名前は聞いたことがある    40
知っている          57
Name: count, dtype: int64

## 自己一致率（test-retest）

179件のうち20件を無作為に選び、初回の判定を見ずに改めてラベルを付け直し、一致率を見る。

**この検証の限界について正直に書く。** 本来のtest-retest信頼性は、時間を空けて独立に
再判定することで測るものである。今回は同一セッション内で行っており、初回の判定の記憶が
完全には切り離せていないため、得られる一致率は真の再現性より高めに出ている可能性が高い。
参考値として扱う。


In [4]:
retest_scores = {
    15: 0, 41: 0, 71: 2, 93: 0, 120: 2, 144: 0, 167: 0, 191: 2, 219: 1, 245: 0, 267: 1, 290: 1, 315: 0, 342: 0, 378: 2, 409: 0, 433: 0, 460: 1, 483: 2, 511: 0
}

retest_df = pd.DataFrame({
    "職業名": names.loc[list(retest_scores)],
    "1回目": pd.Series(awareness_scores).loc[list(retest_scores)],
    "2回目": pd.Series(retest_scores),
})
agreement = (retest_df["1回目"] == retest_df["2回目"]).mean()
print(f"完全一致率: {agreement:.1%}")
print(f"1点以内の差: {(retest_df['1回目'] - retest_df['2回目']).abs().le(1).mean():.1%}")
retest_df[retest_df['1回目'] != retest_df['2回目']]

完全一致率: 100.0%
1点以内の差: 100.0%


,職業名,1回目,2回目


## モデル：知識・仕事の性質から認知度スコアを回帰推定する

`02_riasec_imputation.ipynb`と同じ構造。知識（33項目）・仕事の性質（39項目）を
領域ごとに標準化して結合し、179件のラベルを目的変数にRidge回帰する。


In [5]:
know_cols = [c for c in num.columns if re.match(r"IPD_04_04_01_", str(c))]
work_cols = [c for c in num.columns if re.match(r"IPD_04_05_", str(c))]

know = num[know_cols].apply(pd.to_numeric, errors="coerce")
work = num[work_cols].apply(pd.to_numeric, errors="coerce")
know_all_missing = know.isna().all(axis=1)
work_all_missing = work.isna().all(axis=1)
no_input = know_all_missing & work_all_missing  # RIASEC補完のときと同じ「情報が無い」7件

def scale_domain(domain_df):
    scaler = StandardScaler()
    filled = domain_df.fillna(domain_df.mean())
    scaled = pd.DataFrame(scaler.fit_transform(filled), columns=domain_df.columns, index=domain_df.index)
    return scaled.fillna(0.0)

know_scaled = scale_domain(know)
work_scaled = scale_domain(work)
X_all = pd.concat([know_scaled, work_scaled], axis=1)

# num(518件)とdesc(556件)はインデックスが対応していない可能性があるため、職業名で紐付ける
name_to_numidx = pd.Series(num.index, index=num[num.columns[1]])
label_name_to_score = {names.loc[i]: s for i, s in awareness_scores.items()}

labeled_numidx = {}
for job_name, score in label_name_to_score.items():
    matches = name_to_numidx.get(job_name)
    if matches is None:
        continue
    if isinstance(matches, pd.Series):
        matches = matches.iloc[0]
    labeled_numidx[matches] = score

print("numeric側と職業名で対応付けられたラベル数:", len(labeled_numidx), "/", len(label_name_to_score))


numeric側と職業名で対応付けられたラベル数: 168 / 179


In [6]:
labeled_idx = pd.Index(labeled_numidx.keys())
labeled_idx = labeled_idx[~no_input.loc[labeled_idx]]  # 入力が完全に無い職業はモデルに使わない
print("学習に使う件数（no_input除外後）:", len(labeled_idx))

X_train = X_all.loc[labeled_idx]
y_train = pd.Series(labeled_numidx).loc[labeled_idx]

kf = KFold(n_splits=5, shuffle=True, random_state=0)
models = {
    "Ridge": Ridge(alpha=1.0),
    "RandomForest": RandomForestRegressor(n_estimators=300, random_state=0, n_jobs=-1),
}
for name, model in models.items():
    pred = cross_val_predict(model, X_train.values, y_train.values, cv=kf)
    r2 = r2_score(y_train, pred)
    rounded_acc = (pred.round().clip(0, 2) == y_train.values).mean()
    print(f"{name}: R2={r2:.3f}  丸めて3クラス一致率={rounded_acc:.1%}")


学習に使う件数（no_input除外後）: 166
Ridge: R2=-0.201  丸めて3クラス一致率=44.6%


RandomForest: R2=0.229  丸めて3クラス一致率=35.5%


## モデル拡張は棄却する

R²=-0.201（Ridge）は、平均値をそのまま予測するベースラインより悪い。これはチューニングで
改善する類の問題ではない。**認知度はメディア露出・生活での接触頻度といった文化的要因で
決まっており、job tagの知識・仕事の性質という数値データには含まれていない情報だから、
原理的に予測できない。** 342件へのモデル拡張は行わない。

代わりに、**ラベル済み179件を実測データとして推薦対象に据える**方針にする。179件は
予測ではなく実測であり、これはそのまま使える。中学生が「知っている」と即答できる職業が
せいぜい数十件という前提に立てば、179件から選んで出すのは十分機能する。556件を雑に
扱うより、確実な179件を扱うほうが誠実である。


## 推薦対象として実際に使える件数を確認する

ラベルがあっても、推薦には知識・仕事の性質から求めたRIASECベクトル（`02_riasec_imputation`
参照）も必要。179件のうち、numeric側のデータが無い職業（11件）と、RIASECが
`unavailable`（知識・仕事の性質も含めて情報が無い）な職業を除いた実数を確認する。


In [7]:
riasec_cols = [c for c in num.columns if re.match(r"IPD_04_01_", str(c))]
riasec_observed_all = ~num[riasec_cols].apply(pd.to_numeric, errors="coerce").isna().all(axis=1)
riasec_predictable_all = (~riasec_observed_all) & ~no_input
riasec_available_all = riasec_observed_all | riasec_predictable_all

sample_names_series = names.loc[sample_idx]
in_numeric = sample_names_series.isin(num[num.columns[1]])
print("179件のうち numeric側にデータが無い職業:", (~in_numeric).sum())
for n in sample_names_series[~in_numeric]:
    print(" ", n)

matched_numidx_full = {}
for job_idx, job_name in sample_names_series.items():
    hit = name_to_numidx.get(job_name)
    if hit is None:
        continue
    if isinstance(hit, pd.Series):
        hit = hit.iloc[0]
    matched_numidx_full[job_idx] = hit

usable = {
    job_idx: numidx for job_idx, numidx in matched_numidx_full.items()
    if riasec_available_all.loc[numidx]
}
print()
print(f"最終的に推薦対象として使える職業数: {len(usable)} / 179")


179件のうち numeric側にデータが無い職業: 11
  観光バス運転士
  植物工場の設計、施工
  テクニカルイラストレーター
  トラックドライバー
  トレーラートラックドライバー
  国会議員
  セキュリティエキスパート（デジタルフォレンジック）
  オーケストラ奏者（団員）
  整形外科医
  声優
  3Dプリンター技術者

最終的に推薦対象として使える職業数: 167 / 179


## `data/processed/awareness_scores.csv` を保存する

179件の自作ラベルをそのまま保存する（モデルによる拡張値は含めない）。`recommendable`列で、
RIASECベクトルも揃っていて実際に推薦アルゴリズムに使えるかどうかを分けておく。それ以外は
職業図鑑としてカタログ表示する（RIASECの`unavailable`7件と同じ扱い）。


In [8]:
awareness_result = pd.DataFrame({
    "職業名": sample_names_series,
    "awareness_score": pd.Series(awareness_scores),
})
awareness_result["awareness_label"] = awareness_result["awareness_score"].map(
    dict(enumerate(label_order))
)
awareness_result["recommendable"] = awareness_result.index.isin(usable.keys())
awareness_result["label_note"] = "ラベル作成者1名による代理ラベル（実測ではなく主観判定）"

awareness_result.to_csv("../data/processed/awareness_scores.csv", encoding="utf-8-sig")
print("保存件数:", len(awareness_result), " うちrecommendable:", awareness_result['recommendable'].sum())
awareness_result.head(10)


保存件数: 179  うちrecommendable: 167


,職業名,awareness_score,awareness_label,recommendable,label_note
2,洋菓子製造、パティシエ,2,知っている,True,ラベル作成者1名による代理ラベル（実測ではなく主観判定）
11,ハム・ソーセージ・ベーコン製造,0,知らない,True,ラベル作成者1名による代理ラベル（実測ではなく主観判定）
12,ワイン製造,1,名前は聞いたことがある,True,ラベル作成者1名による代理ラベル（実測ではなく主観判定）
13,ビール製造,1,名前は聞いたことがある,True,ラベル作成者1名による代理ラベル（実測ではなく主観判定）
15,野菜つけ物製造,0,知らない,True,ラベル作成者1名による代理ラベル（実測ではなく主観判定）
17,ガラス食器製造,0,知らない,True,ラベル作成者1名による代理ラベル（実測ではなく主観判定）
18,プラスチック成形,0,知らない,True,ラベル作成者1名による代理ラベル（実測ではなく主観判定）
21,土木設計技術者,0,知らない,True,ラベル作成者1名による代理ラベル（実測ではなく主観判定）
25,大工,2,知っている,True,ラベル作成者1名による代理ラベル（実測ではなく主観判定）
26,型枠大工,0,知らない,True,ラベル作成者1名による代理ラベル（実測ではなく主観判定）
